# 03 City C
**Week 2 only.** The first run must use the code tagged `week1-final`, with no changes.

Write down every problem and fix in `docs/city_c_report.md`.

**Summary (numbers from our run; the cells below recompute them)**

| | MAE, AUD |
|---|---|
| Baseline on Brisbane | 84.5 |
| Model on Sydney + Melbourne (test) | 112.2 (baseline 144.4) |
| Model on Brisbane, first run | 80.3 |
| Model on Brisbane, after changes | 80.3 (no change adopted) |

- The model beats the baseline by 22% on A+B but only by 5% on Brisbane. Brisbane prices are less spread out (so the baseline is already strong there), and the location signal learned in A+B does not transfer.
- Pipeline fixes: none. Model changes: three experiments (distance and density, leave-one-out, host listings count), none adopted.
- Known limitation: the `home` segment is over-priced and the hotel-like segments are under-priced (a price level the model has not seen).

## 1. First run, unchanged
Bronze, silver and validation for Brisbane ran with the `week1-final` code (see `docs/city_c_report.md`, section 1).

## 1b. How city C differs from A and B
Run **after** the unchanged first run (bronze and silver must exist for C). Same profiling code as `01_eda_leakage`, so the comparison is like for like.
Each difference becomes a *pipeline fix* or a *model change* in `docs/city_c_report.md`.

In [0]:
import os, sys
sys.path.append(os.path.abspath("../src"))
import pandas as pd
from pyspark.sql import functions as F
from airbnb import config, eda, features, model, gold

CITY_C = "brisbane"   # must be in config.CITIES and already run through 00_run_pipeline

# Stop early if a city has not been through the pipeline yet (city C: week 2 only)
present = {r["city"] for r in spark.table(config.LISTINGS_TABLE).select("city").distinct().collect()}
missing = [c for c in config.TRAIN_CITIES + [CITY_C] if c not in present]
if missing:
    print(f"No silver rows for {missing}. Run 00_run_pipeline for them first "
          f"(city C only in week 2, after the week1-final tag).")
else:
    cmp = eda.compare_to_training(spark, CITY_C)
    for name, table in cmp.items():
        print(f"=== {name}")
        if isinstance(table, list):
            table = pd.DataFrame({"column": table})
        if len(table) == 0:
            print("  (none)")          # Databricks cannot display an empty table
        else:
            display(table)

## 1c. First run on C: error before any change
The model and `feature_stats.json` come from `02_model` (A and B only) and are used as they are. All in-scope Brisbane listings count as test, because the model has seen none of them.

Run `02_model` first (it writes the A and B gold rows), then these cells. Copy the numbers into `docs/city_c_report.md` before changing anything: `model.save` overwrites the saved model.

In [0]:
fitted, stats = model.load()
d = model.score_new_city(spark, fitted, stats, CITY_C).cache()
first = model.evaluate_new_city(d, CITY_C)
display(first.toPandas().round(1))

In [0]:
# save the first run for the dashboard (stage = 'first_run'); a re-run replaces these rows
gold.write_city_metrics(first, CITY_C, "first_run")
gold.write_city_predictions(d, CITY_C)

### Where the error sits
Bias > 0 means over-pricing. `sc` = in-scope Brisbane listings with the model and baseline predictions.

In [0]:
sc = d.filter("in_scope").withColumn("dist_band_km", F.least(F.floor(F.col("dist_to_centre_km") / 5) * 5, F.lit(30)))
gap = lambda p: F.col(p) - F.col("price")
for col in ("room_type", "property_group", "dist_band_km"):
    display(sc.groupBy(col).agg(F.count("*").alias("n"),
                                F.avg(F.abs(gap("pred"))).alias("model_mae"), F.avg(gap("pred")).alias("model_bias"),
                                F.avg(F.abs(gap("baseline_pred"))).alias("baseline_mae"), F.avg(gap("baseline_pred")).alias("baseline_bias"),
                                ).orderBy(col))

Which property groups carry the error? (`home` is the biggest group and the model is *worse* than the baseline there; hotel-like types are strongly under-priced.)

In [0]:
total = sc.agg(F.sum(F.abs(gap("pred")))).first()[0]
display(sc.groupBy("property_group").agg(F.count("*").alias("n"),
                                         F.round(F.sum(F.abs(gap("pred"))) / total * 100, 1).alias("share_of_error_pct"),
                                         F.round(F.avg(F.abs(gap("pred"))), 1).alias("model_mae"),
                                         F.round(F.avg(F.abs(gap("baseline_pred"))), 1).alias("baseline_mae"),
                                         F.round(F.avg(gap("pred")), 1).alias("model_bias"),
                                         ).orderBy(F.desc("share_of_error_pct")))

The 46 listings 30 km or more from the centre are all on Moreton Island (holiday homes on an island). Distance to the centre means something else there: far is not cheap. They are only ~3% of the total error, so this is a limitation to report, not the main cause.

In [0]:
display(sc.filter("dist_to_centre_km >= 30").select("neighbourhood_cleansed", "property_type", "room_type", "accommodates", "price", "pred", "dist_to_centre_km").orderBy(F.desc("price")))

## 2. Fixes (each one: pipeline fix or model change)
**Pipeline fixes:** none needed (see the report).

**Model changes:** three experiments. Rules for all of them:
- models are fitted on the A+B **train** rows only, and compared on A+B **validation** (as in `02_model`) and on Brisbane;
- the seed alone moves Brisbane MAE by about ±0.5 AUD, so a difference below ~1 AUD is noise;
- a change is adopted only if it helps Brisbane clearly, does not cost much on A+B, and has a reason in the data.

Choosing by the Brisbane result is tuning on city C, so any "after changes" number would be slightly optimistic. We state that in the report.

### Setup
Rebuilds the A+B rows exactly as in `02_model` and the Brisbane rows with the same features. Needs `features.py` with the `dist_pct` column.

In [0]:
import importlib
importlib.reload(features); importlib.reload(model)

ab = model.add_split(spark.table(config.LISTINGS_TABLE).filter(F.col("city").isin(*config.TRAIN_CITIES)))
dev = ab.filter("in_scope AND split != 'test'")
rows = (model.add_baseline(dev, features.make_features(ab, stats))
        .withColumn("label", F.when(F.col("in_scope"), F.log("price"))).filter("in_scope"))
train_df, val_df = rows.filter("split = 'train'"), rows.filter("split = 'val'")
c = features.make_features(model.add_scope(spark.table(config.LISTINGS_TABLE).filter(F.col("city") == CITY_C)), stats).filter("in_scope")

cat, num, est, _ = model.candidates(rows)["Random forest"]

def mae(m, d, city):
    """MAE of the fitted model m on the rows d, for one city ('all' = all cities in d)."""
    return model.evaluate(model.scored(m, d), "pred", "m").filter(F.col("city") == city).first()["mae"]

### Experiment 1: distance and density
**Question:** at 5-20 km from the centre the model over-prices by 15-22 AUD, but at 0-5 km it is unbiased. Does distance to the centre mean something different in Brisbane?

**Result (our run), MAE A+B validation / Brisbane:** as is 120.1 / 81.4; distance as city percentile 119.9 / 81.3; no distance 121.5 / 81.6; no distance and no density 125.1 / 80.5.

**Verdict:** no variant helps Brisbane by more than ~1 AUD, and dropping both features costs 5 AUD (+4%) on A+B. Not adopted. (Distance and density do help in A+B, where location matters a lot.)

In [0]:
variants = {"as is": num,
            "distance as city percentile": ["dist_pct" if x == "dist_to_centre_km" else x for x in num],
            "no distance": [x for x in num if x != "dist_to_centre_km"],
            "no distance, no density": [x for x in num if x not in ("dist_to_centre_km", "density_pct")]}
out = []
for name, cols in variants.items():
    m = model.build(cat, cols, est).fit(train_df)
    out.append({"variant": name, "A+B validation": mae(m, val_df, "all"), "Brisbane": mae(m, c, CITY_C)})
display(pd.DataFrame(out).round(1))

### Experiment 2: leave one feature out
**Question:** is any single feature hurting on Brisbane? Each row drops one feature (`room_type` always stays). Columns `*_change` are against the full model: negative = better.

**Result (our run):** almost every feature helps or is neutral on Brisbane. `has_reviews` has no effect (-0.3 / +0.3). `host_tenure_months` is the most useful in Brisbane (+4.0 when dropped). Only `calculated_host_listings_count` improved Brisbane (-1.9, at +0.8 on A+B). There is a reason in the data (Brisbane has 2.3 listings per host against 1.8 and 2.1, and a median host listing count of 6 against 4), so it goes to experiment 3.

In [0]:
out = []
for drop in [None, "property_group"] + num:
    m = model.build([x for x in cat if x != drop], [x for x in num if x != drop], est).fit(train_df)
    out.append({"dropped": drop or "(none)", "ab_val": mae(m, val_df, "all"), "brisbane": mae(m, c, CITY_C)})
loo = pd.DataFrame(out)
loo["ab_change"] = loo.ab_val - loo.ab_val[0]
loo["brisbane_change"] = loo.brisbane - loo.brisbane[0]
display(loo.round(1))

### Experiment 3: host listings count, three seeds
**Question:** is the -1.9 from experiment 2 real, or seed noise?

**Result (our run), mean of 3 seeds, A+B validation / Brisbane:** as is 119.8 / 79.8; without `calculated_host_listings_count` 120.6 / 79.6.

**Verdict:** the gain is 0.2 AUD on Brisbane (worse on seed 3) and the cost is 0.8 AUD on A+B on every seed. The -1.9 was noise (the seed-42 baseline happened to be 1.5 AUD above the other seeds). Not adopted.

In [0]:
out = []
for seed in (1, 2, 3):
    e = est.copy({est.seed: seed})
    for name, cols in {"as is": num, "no host listings count": [x for x in num if x != "calculated_host_listings_count"]}.items():
        m = model.build(cat, cols, e).fit(train_df)
        out.append({"seed": seed, "variant": name, "ab_val": mae(m, val_df, "all"), "brisbane": mae(m, c, CITY_C)})
seeds = pd.DataFrame(out).sort_values(["variant", "seed"])
display(seeds.round(1))
print(seeds.groupby("variant")[["ab_val", "brisbane"]].mean().round(1))

### Decision
No model change is adopted, so the model on Brisbane after changes is the first-run model (80.3). Nothing is saved as `after_changes`; for the dashboard the final result for Brisbane is `stage = 'first_run'`.

What did not transfer, and could not be fixed without Brisbane prices (a price level the model has not seen):
- `home` (30% of listings, ~37% of the error): over-priced by ~21 AUD, worse than the baseline;
- hotel-like types (hotel, boutique hotel, serviced apartment, loft, villa): under-priced by 53-170 AUD;
- Moreton Island holiday homes: under-priced, because distance means something else there.

## 3. The three numbers: baseline on C, model on A and B, model on C (before and after fixes)

In [0]:
metrics = spark.table(config.METRICS_TABLE).toPandas()
assert not metrics.empty and "stage" in metrics.columns, "model_metrics is empty or has no `stage`: run 02_model, then the save cell in 1c"
display(metrics.sort_values(["stage", "city", "model"]).round(1))

get = lambda stage, city, name: metrics[(metrics.stage == stage) & (metrics.city == city) & (metrics.model == name)].mae.iloc[0]
rows3 = []
for label, stage, city in [("Sydney + Melbourne (test)", "test", "all"), ("Brisbane (first run = final)", "first_run", CITY_C)]:
    b, m_ = get(stage, city, "baseline"), get(stage, city, "model")
    rows3.append({"data": label, "baseline MAE": b, "model MAE": m_, "model better than baseline, %": (1 - m_ / b) * 100})
display(pd.DataFrame(rows3).round(1))